In [ ]:
###
# セットアップ
###

from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")
except Exception:
    ROOT_PATH = Path.cwd()

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

LOCAL_DATA_DIR = ROOT_PATH / "data" / "cats_vs_dogs"
if not LOCAL_DATA_DIR.exists():
    fallback = Path("/content/data/cats_vs_dogs")
    if fallback.exists():
        LOCAL_DATA_DIR = fallback

print("ROOT_PATH:", ROOT_PATH)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)


In [ ]:
###
# 1. 必要なライブラリのインポート
###

import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torchvision import transforms

# GPUが使える場合はGPUを、使えない場合はCPUを使用する設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用するデバイス: {device}")


In [ ]:
###
# 2. データ拡張つきのDataset / DataLoader
###

_FRAC_MAP = {"small": 0.2, "medium": 0.5, "large": 1.0}

class CDDataset(Dataset):
    def __init__(self, df, data_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.data_dir, row["filepath"])).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(row["label"])


TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def get_dc_dataloaders(data_dir, data_size="small", batch_size=32):
    frac = _FRAC_MAP.get(data_size)
    if frac is None:
        raise ValueError("data_size must be one of small, medium, large")

    df = pd.read_csv(os.path.join(data_dir, "labels.csv"))

    def make_loader(split, transform, shuffle):
        split_df = df[df["split"] == split].sample(frac=frac, random_state=61)
        dataset = CDDataset(split_df, data_dir, transform=transform)
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=2,
            pin_memory=torch.cuda.is_available(),
        )

    train_loader = make_loader("train", TRAIN_TRANSFORM, True)
    val_loader = make_loader("val", EVAL_TRANSFORM, False)
    test_loader = make_loader("test", EVAL_TRANSFORM, False)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_dc_dataloaders(
    data_dir=LOCAL_DATA_DIR,
    data_size="small",
    batch_size=32,
)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))


In [ ]:
###
# 3. データ拡張の確認
###

label_names = {0: "cat", 1: "dog"}

def denormalize(image_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)
    image = image_tensor.cpu() * std + mean
    return image.clamp(0, 1)

preview_df = pd.read_csv(os.path.join(LOCAL_DATA_DIR, "labels.csv"))
preview_df = preview_df[preview_df["split"] == "train"]
preview_df = preview_df.sample(n=1, random_state=61)
preview_dataset = CDDataset(preview_df, LOCAL_DATA_DIR, transform=TRAIN_TRANSFORM)

base_image = Image.open(os.path.join(LOCAL_DATA_DIR, preview_df.iloc[0]["filepath"])).convert("RGB")
augmented_images = [preview_dataset[0][0] for _ in range(4)]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(base_image)
axes[0].set_title(f"original\n{label_names[int(preview_df.iloc[0]['label'])]}")
axes[0].axis("off")

for i, image_tensor in enumerate(augmented_images, start=1):
    axes[i].imshow(denormalize(image_tensor).permute(1, 2, 0))
    axes[i].set_title(f"augmented {i}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
###
# 4. CNNの実装
###

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(in_features=128 * 8 * 8, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=2)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = self.conv4(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x


model = SimpleCNN().to(device)
print(model)


In [ ]:
###
# 5. 学習の前準備
###

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20
train_loss_list, val_loss_list, val_acc_list = [], [], []

best_val_loss = float("inf")

os.makedirs(ROOT_PATH / "models", exist_ok=True)
save_path = ROOT_PATH / "models" / "11_data_augmentation.pth"


In [ ]:
###
# 6. 学習ループの実装
###

print("学習開始")
for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    running_train_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Train ", leave=False)
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    epoch_train_loss = running_train_loss / len(train_loader)
    train_loss_list.append(epoch_train_loss)

    # --- Validation ---
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Val ", leave=False)
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            val_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_val_acc = 100 * correct / total

    val_loss_list.append(epoch_val_loss)
    val_acc_list.append(epoch_val_acc)

    mark = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), save_path)
        mark = " [saved]"

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_acc:.2f}% {mark}"
    )

print("学習完了")


In [ ]:
###
# 7. 結果の描画 (グラフ)
###

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_loss_list, label='Train Loss', color='blue', marker='o')
ax1.plot(val_loss_list, label='Validation Loss', color='orange', marker='o')
ax1.set_title('Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(val_acc_list, label='Validation Accuracy', color='green', marker='o')
ax2.set_title('Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()
